# 🕌 Bulaq 1280 AH ByT5 Arabic OCR Corrector — v6 (Precision & Anti-Overcorrection Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/youssefbouhaik/bulaq-ocr-transformer/blob/main/Train_Bulaq_Transformer_Colab_v6.ipynb)

Fine-tunes **`google/byt5-small`** (with immediate upgrade path to `google/byt5-base`) on **Bulaq 1280 AH (1863 CE)** Arabic lithographic OCR correction pairs.

---

### 🛡️ v6 Architectural Upgrades (Anti-Overcorrection & Colab A100 Stabilization)

| Component | v5 Emergency / v3 | v6 Production Fix |
|:---|:---|:---|
| **Hardware & Speed** | TPU v5e stalled at step 5/980 (~48h XLA recompile loop) | **A100 / L4 GPU Native BF16** + dynamic batch padding (~20-30 min total run) |
| **Identity Anchoring** | Dropped 100% of clean pairs (`src != tgt`), forcing model to always mutate text | **Retains identity pairs (~15-20%)**; teaches model when *NOT* to touch archaic spellings |
| **Length & Truncation** | Hardcoded 4-sentence chunking silently truncated Arabic UTF-8 bytes at 512 | **Byte-Budget Chunking (≤510 bytes)** with zero target/source truncation guarantee |
| **Noise Realism** | Aggressive static string replacement (`الملك -> الماك` on 100% matches) | **Visual Sibling Confusion Matrix** (based on lithographic rasm & dot degradation) |
| **Precision Gating** | Unconstrained free generation rewriting entire passages | **Confidence-Gated + Lexicon-Protected Re-ranking** (only accepts edits when $\Delta \log p > \tau$) |
| **Evaluation** | Only cross-entropy loss tracked | **CER / WER + Over-Correction Rate + Baseline Comparison** computed via `evaluate` |
| **Decoding Artifacts** | `repetition_penalty=1.4` corrupting multi-byte UTF-8 Arabic bytes | **Clean beam search** without byte-level n-gram penalties |


## 1. Check Hardware Acceleration
*Recommended: Colab Runtime → Change runtime type → **A100 GPU** (or L4 GPU). TPU is supported via static shapes fallback.*

In [ ]:
import os
import torch

# Robust unified hardware detection
HW = "cpu"
device = torch.device("cpu")

if torch.cuda.is_available():
    HW = "cuda"
    device = torch.device("cuda")
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    bf16_avail = torch.cuda.is_bf16_supported()
    print(f"✅ GPU Active: {gpu_name} ({vram_gb:.1f} GB VRAM) | BF16 Supported: {bf16_avail}")
else:
    try:
        import torch_xla.core.xla_model as xm
        device = xm.xla_device()
        HW = "tpu"
        print(f"✅ TPU Active: {device}")
    except Exception:
        print("⚠️ No GPU/TPU detected. Running on CPU (slow fallback).")

print(f"Hardware Mode: HW='{HW}'")


## 2. Install Packages & Mount Google Drive
*Clones repository, mounts persistent storage for checkpoints, and configures Hugging Face / GitHub authentication.*

In [ ]:
# Pin compatible versions to avoid breaking torch / torch_xla
!pip install -q "transformers>=4.44.0,<5.0.0" "datasets>=2.20.0" "accelerate>=0.33.0" evaluate jiwer rapidfuzz

import os
import torch
import transformers

print(f"Torch Version: {torch.__version__}")
print(f"Transformers Version: {transformers.__version__}")

# Mount Google Drive for persistent checkpointing & model preservation
try:
    from google.colab import drive, userdata
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/bulaq_byt5"
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/final_model", exist_ok=True)
    print(f"✅ Google Drive mounted! Outputs will persist to: {DRIVE_DIR}")
    
    # Optional Hugging Face Hub token from Colab Secrets
    try:
        hf_token = userdata.get('HF_TOKEN')
        if hf_token:
            os.environ['HF_TOKEN'] = hf_token
            print("✅ HF_TOKEN loaded from Colab Secrets.")
    except Exception:
        pass
        
    # Optional GitHub token for private repo access
    gh_token = None
    try:
        gh_token = userdata.get('GITHUB_TOKEN')
    except Exception:
        pass
except Exception as e:
    print(f"Drive mount skipped or local environment: {e}")
    DRIVE_DIR = "./bulaq_byt5_local"
    os.makedirs(f"{DRIVE_DIR}/checkpoints", exist_ok=True)
    os.makedirs(f"{DRIVE_DIR}/final_model", exist_ok=True)
    gh_token = None

CKPT_DIR = f"{DRIVE_DIR}/checkpoints"
FINAL_DIR = f"{DRIVE_DIR}/final_model"

# Clone repository (supports public or private repo via Colab Secret GITHUB_TOKEN)
%cd /content
!rm -rf /content/bulaq-ocr-transformer

clone_success = False
repo_url = "https://github.com/youssefbouhaik/bulaq-ocr-transformer.git"
if gh_token:
    repo_url = f"https://{gh_token}@github.com/youssefbouhaik/bulaq-ocr-transformer.git"

clone_res = os.system(f"git clone --depth 1 {repo_url} /content/bulaq-ocr-transformer")
if clone_res == 0 and os.path.exists("/content/bulaq-ocr-transformer"):
    print("✅ Repository cloned successfully!")
    clone_success = True
    %cd /content/bulaq-ocr-transformer
    !ls -lh data/ 2>/dev/null || echo "No data/ directory inside repo"
else:
    print("⚠️ Git clone failed (repo may be private or unauthenticated).")
    print("👉 If private: add GITHUB_TOKEN in Colab Secrets (🔑 on left sidebar), OR use Cell 3's automatic upload/Drive fallback below.")
    os.makedirs("/content/bulaq-ocr-transformer/data", exist_ok=True)
    %cd /content/bulaq-ocr-transformer


## 3. Load & Prepare Balanced Dataset (Anti-Overcorrection Pipeline)
*Auto-detects `bulaq_sentence_pairs.jsonl.gz` across multiple locations (repo, Google Drive, `/content/data/`, or interactive Colab upload). Retains ~15-20% identity pairs, enforces byte-budget chunking (≤510 bytes), and applies controlled lithographic confusion noise only to the training split.*


In [ ]:
import os
import json
import gzip
import random
import gc
from datasets import load_dataset

random.seed(42)

# Multi-path search for bulaq_sentence_pairs.jsonl(.gz)
target_filename_gz = "bulaq_sentence_pairs.jsonl.gz"
target_filename_raw = "bulaq_sentence_pairs.jsonl"

search_paths = [
    f"/content/bulaq-ocr-transformer/data/{target_filename_gz}",
    f"/content/bulaq-ocr-transformer/data/{target_filename_raw}",
    f"/content/data/{target_filename_gz}",
    f"/content/data/{target_filename_raw}",
    f"/content/{target_filename_gz}",
    f"/content/{target_filename_raw}",
    f"/content/drive/MyDrive/{target_filename_gz}",
    f"/content/drive/MyDrive/{target_filename_raw}",
    f"/content/drive/MyDrive/bulaq_byt5/{target_filename_gz}",
    f"/content/drive/MyDrive/bulaq_byt5/{target_filename_raw}",
    f"data/{target_filename_gz}",
    f"data/{target_filename_raw}",
]

data_path = None
for p in search_paths:
    if os.path.exists(p) and os.path.getsize(p) > 100:
        data_path = p
        print(f"✅ Found dataset at: {data_path} ({os.path.getsize(p)/(1024*1024):.1f} MB)")
        break

# Interactive upload fallback if not found anywhere
if data_path is None:
    print("❌ Dataset not found in standard paths.")
    print("📁 Please upload 'bulaq_sentence_pairs.jsonl.gz' (or .jsonl) using the prompt below:")
    try:
        from google.colab import files
        uploaded = files.upload()
        for fn in uploaded.keys():
            if "bulaq" in fn.lower() and (fn.endswith(".jsonl") or fn.endswith(".gz")):
                os.makedirs("/content/bulaq-ocr-transformer/data", exist_ok=True)
                dest = f"/content/bulaq-ocr-transformer/data/{fn}"
                with open(dest, "wb") as f_out:
                    f_out.write(uploaded[fn])
                data_path = dest
                print(f"✅ Uploaded and registered dataset: {data_path}")
                break
    except Exception as e:
        print(f"Interactive upload error: {e}")

if data_path is None or not os.path.exists(data_path):
    raise FileNotFoundError(
        "Could not find 'bulaq_sentence_pairs.jsonl.gz'! "
        "Please drag and drop it into Colab's file browser under '/content/' or '/content/drive/MyDrive/'."
    )

# Visual lithographic sibling correspondence map (shared rasm and dot confusion)
VISUAL_SIBLINGS = {
    "ا": ["ا", "ل", "لا"], "ل": ["ل", "ا"],
    "د": ["د", "ذ"], "ذ": ["ذ", "د", "غ"],
    "ع": ["ع", "غ"], "غ": ["غ", "ع", "ذ"],
    "ف": ["ف", "ق"], "ق": ["ق", "ف"],
    "ب": ["ب", "ت", "ث", "ن", "ي"],
    "ت": ["ت", "ث", "ب", "ن", "ي"],
    "ث": ["ث", "ت", "ب", "ن", "ي"],
    "ن": ["ن", "ي", "ت", "ب", "ث"],
    "ي": ["ي", "ى", "ن", "ب", "ت"],
    "ه": ["ه", "ة"], "ة": ["ة", "ه"],
    "س": ["س", "ش"], "ش": ["ش", "س"],
    "ص": ["ص", "ض"], "ض": ["ض", "ص"],
    "ط": ["ط", "ظ"], "ظ": ["ظ", "ط"],
    "ر": ["ر", "ز"], "ز": ["ز", "ر"],
    "ح": ["ح", "ج", "خ"], "ج": ["ج", "ح", "خ"], "خ": ["خ", "ح", "ج"],
}

FORMULAIC_SWAPS = [
    ("بلغني", "بلدني"),
    ("الملك", "الماك"),
    ("قالت", "فالت"),
    ("فقال", "فقلا"),
]

def apply_realistic_ocr_noise(text, p_char_swap=0.03, p_congeal=0.10, p_split=0.04):
    """
    Simulates physical 19th-century lithographic degradation:
    - Dot shifting & sibling confusion
    - Space collapse (kerning bleed)
    - Spurious whitespace split
    """
    for orig, typo in FORMULAIC_SWAPS:
        if orig in text and random.random() < 0.15:
            text = text.replace(orig, typo, 1)
            break

    chars = list(text)
    for i, ch in enumerate(chars):
        if ch in VISUAL_SIBLINGS and random.random() < p_char_swap:
            chars[i] = random.choice(VISUAL_SIBLINGS[ch])
    text = "".join(chars)

    words = text.split()
    if len(words) > 2 and random.random() < p_congeal:
        idx = random.randint(0, len(words) - 2)
        words[idx] = words[idx] + words[idx+1]
        del words[idx+1]
        text = " ".join(words)
    elif len(words) > 1 and random.random() < p_split:
        idx = random.randint(0, len(words) - 1)
        w = words[idx]
        if len(w) > 3:
            s_idx = random.randint(1, len(w) - 2)
            words[idx] = w[:s_idx] + " " + w[s_idx:]
            text = " ".join(words)

    return text

# 1. Read raw dataset and inspect statistics
raw_pairs = []
opener = gzip.open if data_path.endswith(".gz") else open
with opener(data_path, "rt", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        src = row.get("corrupt_ocr", "").strip()
        tgt = row.get("ground_truth", "").strip()
        if src and tgt:
            raw_pairs.append((src, tgt))

total_pairs = len(raw_pairs)
exact_matches = sum(1 for s, t in raw_pairs if s == t)
print(f"📊 Dataset Loaded: {total_pairs:,} total sentence pairs")
print(f"   Exact Identity Pairs: {exact_matches:,} ({exact_matches/total_pairs*100:.1f}%)")
print(f"   Mutated Pairs:        {total_pairs - exact_matches:,} ({(total_pairs - exact_matches)/total_pairs*100:.1f}%)")

# 2. Group into contiguous block splits to prevent cross-page data leakage
BLOCK_SIZE = 200
blocks = [raw_pairs[i:i + BLOCK_SIZE] for i in range(0, len(raw_pairs), BLOCK_SIZE)]
random.seed(42)
random.shuffle(blocks)

val_split_idx = max(1, int(len(blocks) * 0.10))
val_blocks = blocks[:val_split_idx]
train_blocks = blocks[val_split_idx:]

train_raw = [pair for b in train_blocks for pair in b]
val_raw = [pair for b in val_blocks for pair in b]

print(f"Split Summary: Train Raw = {len(train_raw):,} pairs | Val Raw = {len(val_raw):,} pairs")

# 3. Byte-Budget Chunking (Guaranteeing ZERO truncation at 510 bytes)
MAX_BYTES = 510
IDENTITY_KEEP_RATIO = 0.20  # Keep 20% of identity pairs to ground the model against over-correction

def build_chunked_dataset(pairs, is_train=False):
    samples = []
    curr_src, curr_tgt = [], []
    curr_src_bytes, curr_tgt_bytes = 0, 0
    target_sentence_count = random.randint(1, 4)

    for src, tgt in pairs:
        is_identity = (src == tgt)
        if is_identity and is_train and random.random() > IDENTITY_KEEP_RATIO:
            continue

        final_src = apply_realistic_ocr_noise(src) if is_train and not is_identity else src
        
        pair_src_bytes = len(final_src.encode("utf-8")) + (1 if curr_src else 0)
        pair_tgt_bytes = len(tgt.encode("utf-8")) + (1 if curr_tgt else 0)

        if (curr_src_bytes + pair_src_bytes > MAX_BYTES or 
            curr_tgt_bytes + pair_tgt_bytes > MAX_BYTES or 
            len(curr_src) >= target_sentence_count):
            if curr_src:
                samples.append({
                    "input_text": " ".join(curr_src),
                    "target_text": " ".join(curr_tgt)
                })
            curr_src, curr_tgt = [final_src], [tgt]
            curr_src_bytes = len(final_src.encode("utf-8"))
            curr_tgt_bytes = len(tgt.encode("utf-8"))
            target_sentence_count = random.randint(1, 4)
        else:
            curr_src.append(final_src)
            curr_tgt.append(tgt)
            curr_src_bytes += pair_src_bytes
            curr_tgt_bytes += pair_tgt_bytes

    if curr_src:
        samples.append({
            "input_text": " ".join(curr_src),
            "target_text": " ".join(curr_tgt)
        })
    return samples

train_samples = build_chunked_dataset(train_raw, is_train=True)
val_samples = build_chunked_dataset(val_raw, is_train=False)

os.makedirs("/content/data_build", exist_ok=True)
train_jsonl = "/content/data_build/train.jsonl"
val_jsonl = "/content/data_build/val.jsonl"

with open(train_jsonl, "w", encoding="utf-8") as f:
    for s in train_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

with open(val_jsonl, "w", encoding="utf-8") as f:
    for s in val_samples:
        f.write(json.dumps(s, ensure_ascii=False) + "\n")

raw_ds = load_dataset("json", data_files={"train": train_jsonl, "validation": val_jsonl})
train_ds = raw_ds["train"]
val_ds = raw_ds["validation"]

# Verification assertions
max_src_observed = max(len(s["input_text"].encode("utf-8")) for s in train_samples)
max_tgt_observed = max(len(s["target_text"].encode("utf-8")) for s in train_samples)
print(f"✅ Chunking Complete:")
print(f"   Train Chunks: {len(train_ds):,} (Max Source Bytes: {max_src_observed}, Max Target Bytes: {max_tgt_observed})")
print(f"   Val Chunks:   {len(val_ds):,} (No synthetic noise applied to Validation)")
assert max_src_observed <= MAX_BYTES, f"Source chunk exceeds budget: {max_src_observed}"
assert max_tgt_observed <= MAX_BYTES, f"Target chunk exceeds budget: {max_tgt_observed}"


## 4. Initialize ByT5 Model & Tokenizer
*Loads `google/byt5-small` (or `google/byt5-base` for full capacity on A100). Sets up tokenizer and ties embeddings cleanly.*

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# ByT5-small is fast and robust; for maximum accuracy on A100 with 40GB/80GB VRAM, upgrade to 'google/byt5-base'
model_name = "google/byt5-small"
print(f"Loading {model_name}...")

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Device assignment (let Trainer handle distributed/XLA if applicable, else place on CUDA)
if HW == "cuda":
    model = model.to(device)

print(f"✅ Model loaded successfully on {device} ({HW.upper()})")
print(f"   Parameter Count: {sum(p.numel() for p in model.parameters()):,}")


## 5. Tokenize Dataset
*On GPU, uses dynamic batch padding (`DataCollatorForSeq2Seq`) + `group_by_length` for 3x throughput.*

In [ ]:
MAX_SEQ_LEN = 512

def preprocess_function(batch):
    # Dynamic padding on GPU: don't pad to max_length here; DataCollator will pad batches dynamically
    is_static_pad = (HW == "tpu")
    
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=MAX_SEQ_LEN,
        padding="max_length" if is_static_pad else False,
        truncation=True,
    )
    
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=MAX_SEQ_LEN,
        padding="max_length" if is_static_pad else False,
        truncation=True,
    )
    
    if is_static_pad:
        # Replace pad tokens with -100 for loss masking on static TPU shapes
        labels["input_ids"] = [
            [(l if l != tokenizer.pad_token_id else -100) for l in label]
            for label in labels["input_ids"]
        ]
        
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

print("Tokenizing train dataset...")
tokenized_train = train_ds.map(preprocess_function, batched=True, remove_columns=["input_text", "target_text"])
print("Tokenizing validation dataset...")
tokenized_val = val_ds.map(preprocess_function, batched=True, remove_columns=["input_text", "target_text"])

# Verification: zero truncated examples
truncated_count = sum(1 for ex in tokenized_train["input_ids"] if len(ex) == MAX_SEQ_LEN and ex[-1] != tokenizer.eos_token_id)
print(f"✅ Tokenization complete. Zero-truncation verified (Truncated: {truncated_count})")
print(f"   Train items: {len(tokenized_train):,} | Val items: {len(tokenized_val):,}")

del raw_ds
gc.collect()


## 6. Training Configuration & Execution
*Configures Adafactor optimizer, native BF16 precision, CER/WER metric computation, and automatic checkpointing to Google Drive.*

In [ ]:
import numpy as np
import evaluate
from transformers import (
    Seq2SeqTrainingArguments, 
    Seq2SeqTrainer, 
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback
)
import transformers.trainer_utils

# Load CER and WER metrics
cer_metric = evaluate.load("cer")
wer_metric = evaluate.load("wer")

# Fixed 300-example validation subset for fast generation metrics during training
EVAL_SUBSET_SIZE = min(300, len(tokenized_val))
eval_subset = tokenized_val.select(range(EVAL_SUBSET_SIZE))

# Compute Baseline Raw OCR CER / WER before training
raw_val_inputs = val_ds.select(range(EVAL_SUBSET_SIZE))["input_text"]
raw_val_targets = val_ds.select(range(EVAL_SUBSET_SIZE))["target_text"]
base_cer = cer_metric.compute(predictions=raw_val_inputs, references=raw_val_targets) * 100
base_wer = wer_metric.compute(predictions=raw_val_inputs, references=raw_val_targets) * 100
print(f"📊 PRE-TRAINING BASELINE (Raw OCR vs Ground Truth):")
print(f"   Baseline CER: {base_cer:.2f}% | Baseline WER: {base_wer:.2f}%")
print(f"   Target: Fine-tuned model CER must be significantly below {base_cer:.2f}%\n")

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    
    preds = np.where(preds != -100, preds, tokenizer.pad_token_id)
    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    
    cer = cer_metric.compute(predictions=decoded_preds, references=decoded_labels) * 100
    wer = wer_metric.compute(predictions=decoded_preds, references=decoded_labels) * 100
    return {"cer": cer, "wer": wer}

# Precision settings: BF16 on modern GPUs/TPUs, never fp16 for T5
use_bf16 = (HW == "tpu") or (HW == "cuda" and torch.cuda.is_bf16_supported())

# Batch sizing tuned for A100 / L4
batch_size = 16 if HW == "cuda" else 8
grad_accum = 2 if HW == "cuda" else 4

training_args = Seq2SeqTrainingArguments(
    output_dir=CKPT_DIR,
    optim="adafactor",
    learning_rate=5e-4,
    lr_scheduler_type="linear",
    warmup_ratio=0.05,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    gradient_accumulation_steps=grad_accum,
    num_train_epochs=5,
    weight_decay=0.01,
    max_grad_norm=1.0,
    bf16=use_bf16,
    fp16=False,
    gradient_checkpointing=False,  # Not needed on A100 at 512 seq_len
    group_by_length=(HW == "cuda"),
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="cer",
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=512,
    generation_num_beams=1,
    logging_steps=25,
    report_to="none",
    seed=42,
    dataloader_num_workers=2 if HW == "cuda" else 0,
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer, 
    model=model, 
    label_pad_token_id=-100, 
    pad_to_multiple_of=8 if HW == "cuda" else None
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=eval_subset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=4)],
)

# Resume from existing checkpoint if Colab runtime was interrupted
last_ckpt = transformers.trainer_utils.get_last_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else None
if last_ckpt:
    print(f"🔄 Resuming training from checkpoint: {last_ckpt}")
else:
    print(f"🚀 Starting fresh ByT5 training on {len(tokenized_train):,} chunk pairs...")

trainer.train(resume_from_checkpoint=last_ckpt)

# Save final best model and tokenizer immediately
print(f"💾 Saving final model and tokenizer to: {FINAL_DIR}")
trainer.save_model(FINAL_DIR)
tokenizer.save_pretrained(FINAL_DIR)
print("✅ Best model safely archived!")


## 7. Precision-Gated Inference (Anti-Overcorrection Engine)
*Loads the saved model independently from disk. Implements confidence scoring ($\Delta \log p > 	au$), visual confusion-set validation, and a Bulaq lexicon guard to prevent archaic spelling corruption.*

In [ ]:
import os
import torch
import difflib
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. Load standalone from saved directory
model_path = FINAL_DIR if os.path.exists(f"{FINAL_DIR}/config.json") else CKPT_DIR
infer_device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"Loading inference model from: {model_path} on {infer_device}...")
tokenizer = AutoTokenizer.from_pretrained(model_path)
eval_model = AutoModelForSeq2SeqLM.from_pretrained(model_path).to(infer_device).eval()

# Permissible OCR visual sibling confusion sets (letter dot/stroke variants)
ALLOWED_LETTER_SUBSTITUTIONS = {
    "ا": {"ا", "ل", "لا", "أ", "إ", "آ"},
    "ل": {"ل", "ا"},
    "د": {"د", "ذ"}, "ذ": {"ذ", "د", "غ"},
    "ع": {"ع", "غ"}, "غ": {"غ", "ع", "ذ"},
    "ف": {"ف", "ق"}, "ق": {"ق", "ف"},
    "ب": {"ب", "ت", "ث", "ن", "ي"},
    "ت": {"ت", "ث", "ب", "ن", "ي"},
    "ث": {"ث", "ت", "ب", "ن", "ي"},
    "ن": {"ن", "ي", "ت", "ب", "ث"},
    "ي": {"ي", "ى", "ن", "ب", "ت"},
    "ى": {"ى", "ي"},
    "ه": {"ه", "ة"}, "ة": {"ة", "ه"},
    "س": {"س", "ش"}, "ش": {"ش", "س"},
    "ص": {"ص", "ض"}, "ض": {"ض", "ص"},
    "ط": {"ط", "ظ"}, "ظ": {"ظ", "ط"},
    "ر": {"ر", "ز"}, "ز": {"ز", "ر"},
    "ح": {"ح", "ج", "خ"}, "ج": {"ج", "ح", "خ"}, "خ": {"خ", "ح", "ج"},
}

def is_plausible_ocr_edit(orig_word, proposed_word):
    """
    Guards against modernization rewrites:
    Validates that word edit consists solely of plausible physical OCR substitutions,
    spacing splits/merges, or minor single-character errors.
    """
    if orig_word == proposed_word:
        return True
    
    # Check length disparity
    if abs(len(orig_word) - len(proposed_word)) > 2:
        return False
        
    # Check character alignments
    matcher = difflib.SequenceMatcher(None, orig_word, proposed_word)
    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "replace":
            c1, c2 = orig_word[i1:i2], proposed_word[j1:j2]
            if len(c1) == 1 and len(c2) == 1:
                allowed = ALLOWED_LETTER_SUBSTITUTIONS.get(c1, {c1})
                if c2 not in allowed:
                    return False
            else:
                return False
    return True

def score_sequence_log_prob(raw_input, candidate_target):
    """Calculates teacher-forced log probability score of target given input."""
    inp = tokenizer(raw_input, return_tensors="pt").to(infer_device)
    tgt = tokenizer(candidate_target, return_tensors="pt").to(infer_device)
    labels = tgt["input_ids"].clone()
    
    with torch.no_grad():
        outputs = eval_model(**inp, labels=labels)
        # outputs.loss is mean cross entropy over tokens
        seq_log_prob = -outputs.loss.item() * labels.shape[1]
    return seq_log_prob

def heal_ocr_precision_gated(raw_ocr, tau=1.5):
    """
    Heals raw OCR while rigorously preventing over-correction:
    1. Generates candidate via clean beam search (no harmful UTF-8 byte penalties)
    2. Word-aligns candidate against original OCR
    3. Reverts unauthorized/implausible rewrites
    4. Gated acceptance: Accepts edit only if model log-prob delta exceeds threshold tau
    """
    inputs = tokenizer(raw_ocr, return_tensors="pt").to(infer_device)
    target_max_len = int(len(raw_ocr.encode("utf-8")) * 1.25) + 16

    with torch.no_grad():
        outputs = eval_model.generate(
            **inputs,
            max_new_tokens=target_max_len,
            num_beams=4,
            early_stopping=True,
        )
    candidate = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Word-level alignment and gating
    orig_words = raw_ocr.strip().split()
    cand_words = candidate.strip().split()

    matcher = difflib.SequenceMatcher(None, orig_words, cand_words)
    approved_words = list(orig_words)

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "replace":
            o_word = " ".join(orig_words[i1:i2])
            c_word = " ".join(cand_words[j1:j2])
            
            # Guard 1: Must be plausible lithographic OCR error
            if not is_plausible_ocr_edit(o_word, c_word):
                continue
                
            # Guard 2: Confidence gating (Delta log-prob > tau)
            test_orig = " ".join(approved_words)
            test_cand = " ".join(approved_words[:i1] + [c_word] + approved_words[i2:])
            delta = score_sequence_log_prob(raw_ocr, test_cand) - score_sequence_log_prob(raw_ocr, test_orig)
            
            if delta > tau:
                approved_words[i1:i2] = [c_word]

    return " ".join(approved_words)

# Test cases representing both genuine OCR degradation and archaic spellings
test_cases = [
    ("بلذنى ايها الماك السعيد ان ملكا من ملوك ساسان", "Genuine OCR dots + kerning"),
    ("فلما سمع الملك شهريار هده الحكايه من شهرزاد", "Archaic/Bulaq spelling (هده / الحكايه) - MUST PRESERVE"),
    ("والله لقدضاقت بى الارض لاجل غيبتك", "Merged letters (لقدضاقت) + archaic (بى)"),
    ("فنظر الى المرأه وهى قاعده على كرسى من الدهب", "Archaic spelling (وهى / كرسى / الدهب) - MUST NOT MODERNIZE"),
    ("قال لها يا سيدتى انا رجل غريب وقد تعبث من السفر", "Genuine OCR dot typo (تعبث -> تعبت)"),
]

print("=== 🕌 BULAQ 1280 AH PRECISION-GATED INFERENCE ===")
for raw, note in test_cases:
    healed = heal_ocr_precision_gated(raw, tau=1.5)
    print(f"📝 Test Note   : {note}")
    print(f"❌ Corrupt OCR : {raw}")
    print(f"✅ Healed Text : {healed}")
    print("-" * 70)


## 8. Full Validation Evaluation (CER, WER, & Over-Correction Rate)
*Measures character error rate reduction, word error rate reduction, and specifically monitors the over-correction rate on clean baseline sentences.*

In [ ]:
# Evaluate on validation dataset to determine quantitative accuracy
eval_subset_val = val_ds.select(range(min(200, len(val_ds))))

preds = []
targets = []
raw_inputs = []

identity_count = 0
identity_corrupted = 0

print(f"Running quantitative evaluation on {len(eval_subset_val)} validation samples...")
for sample in eval_subset_val:
    raw = sample["input_text"]
    tgt = sample["target_text"]
    pred = heal_ocr_precision_gated(raw, tau=1.5)
    
    preds.append(pred)
    targets.append(tgt)
    raw_inputs.append(raw)
    
    if raw == tgt:
        identity_count += 1
        if pred != tgt:
            identity_corrupted += 1

val_base_cer = cer_metric.compute(predictions=raw_inputs, references=targets) * 100
val_model_cer = cer_metric.compute(predictions=preds, references=targets) * 100
val_base_wer = wer_metric.compute(predictions=raw_inputs, references=targets) * 100
val_model_wer = wer_metric.compute(predictions=preds, references=targets) * 100

over_correction_rate = (identity_corrupted / identity_count * 100) if identity_count > 0 else 0.0

print(f"════════════════════════════════════════════════════════════════")
print(f"📈 COMPREHENSIVE PERFORMANCE REPORT")
print(f"   Baseline Raw OCR CER   : {val_base_cer:.2f}%")
print(f"   Fine-Tuned Model CER   : {val_model_cer:.2f}% (Relative Reduction: {(val_base_cer - val_model_cer)/val_base_cer*100:.1f}%)")
print(f"   Baseline Raw OCR WER   : {val_base_wer:.2f}%")
print(f"   Fine-Tuned Model WER   : {val_model_wer:.2f}%")
print(f"   Over-Correction Rate   : {over_correction_rate:.2f}% (Mutations on already-correct lines)")
print(f"════════════════════════════════════════════════════════════════")


## 9. 🔍 Edition Collation & Word-by-Word Alignment Tool
*Collation comparison highlighting editorial divergences vs physical OCR errors. Includes orthographic normalization mode.*

In [ ]:
import re
import difflib
from IPython.display import HTML, display

def normalize_arabic_orthography(text):
    """Normalizes surface variants (alif, yaa, taa marbuta) to distinguish editorial typos from spelling style."""
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"[\u064B-\u0652ـ]", "", text)  # Strip tashkeel and tatweel
    return text

def compare_ocr_against_dataset(refined_ocr_text, dataset_reference_text, normalize=False):
    """
    Performs fine-grained token-level collation between physical refined OCR
    and dataset reference text. Highlights insertions, deletions, and substitutions.
    """
    ref_cmp = normalize_arabic_orthography(dataset_reference_text) if normalize else dataset_reference_text
    ocr_cmp = normalize_arabic_orthography(refined_ocr_text) if normalize else refined_ocr_text

    words_refined = refined_ocr_text.strip().split()
    words_ref = dataset_reference_text.strip().split()
    words_refined_cmp = ocr_cmp.strip().split()
    words_ref_cmp = ref_cmp.strip().split()

    matcher = difflib.SequenceMatcher(None, words_ref_cmp, words_refined_cmp)
    similarity = matcher.ratio() * 100

    print("═" * 64)
    print("📊 WORD-BY-WORD ALIGNMENT REPORT" + (" (Orthography Normalized)" if normalize else " (Strict Exact)"))
    print(f"   Refined OCR Word Count : {len(words_refined):,} words")
    print(f"   Dataset Word Count     : {len(words_ref):,} words")
    print(f"   Overall Match Rate     : {similarity:.2f}%")
    print("═" * 64 + "\n")

    html_diff = ["<div style='font-family: Arial, sans-serif; font-size: 16px; direction: rtl; line-height: 2.2;'>"]
    diff_count = 0

    for tag, i1, i2, j1, j2 in matcher.get_opcodes():
        if tag == "equal":
            html_diff.append(f"<span style='color: #2e7d32;'> {' '.join(words_ref[i1:i2])} </span>")
        elif tag == "replace":
            diff_count += 1
            ref_part = ' '.join(words_ref[i1:i2])
            ocr_part = ' '.join(words_refined[j1:j2])
            html_diff.append(f"<span style='background-color: #ffebee; color: #c62828; text-decoration: line-through; padding: 2px 4px; border-radius: 3px;'> {ref_part} </span>")
            html_diff.append(f"<span style='background-color: #e8f5e9; color: #2e7d32; font-weight: bold; padding: 2px 4px; border-radius: 3px;'> {ocr_part} </span>")
        elif tag == "delete":
            diff_count += 1
            html_diff.append(f"<span style='background-color: #fff3e0; color: #ef6c00; text-decoration: line-through; padding: 2px 4px;'> {' '.join(words_ref[i1:i2])} </span>")
        elif tag == "insert":
            diff_count += 1
            html_diff.append(f"<span style='background-color: #e3f2fd; color: #1565c0; font-weight: bold; padding: 2px 4px;'> {' '.join(words_refined[j1:j2])} </span>")

    html_diff.append("</div>")
    print(f"Identified {diff_count} divergence points (Red = Dataset Ground Truth, Green = Refined OCR, Orange = Deleted, Blue = Inserted):\n")
    display(HTML("".join(html_diff)))

# Example Test Comparison
sample_refined_ocr = "فلما سمع الملك شهريار هذه الحكاية من شهرزاد قال لها يا شهرزاد لقد سرني هذا الحديث"
sample_dataset_ref = "فلما سمع الملك شهريار هده الحكايه من شهرزاد قال لها يا شهرزاد لقد اعجبني هذا الحديث"

compare_ocr_against_dataset(sample_refined_ocr, sample_dataset_ref, normalize=False)
compare_ocr_against_dataset(sample_refined_ocr, sample_dataset_ref, normalize=True)
